# RSTC2026 — Vegetation classification with Random Forest
## Adventdalen, Svalbard

This exercise uses the **same small Adventdalen area as Block 1** and **one Sentinel-2 Level-2A acquisition**.

The workflow is deliberately simple:

**Sentinel-2 image → predictors → reference labels → training samples → Random Forest → classified map → validation**

The purpose is to understand what each part of a supervised classification does, rather than to optimise a model.

### Classes

- **0 — Vegetation**
- **1 — Bare / sparsely vegetated ground**
- **2 — Water**

### Predictors

We use a small set of variables that are easy to interpret:

- B2 — Blue reflectance
- B3 — Green reflectance
- B4 — Red reflectance
- B8 — Near-infrared reflectance
- NDVI — a simple vegetation-sensitive index
- Elevation — an environmental predictor from the Copernicus DEM

The elevation model adds environmental context. In Arctic valleys, vegetation distribution is often related not only to spectral response but also to terrain position and elevation.

This is still a simple teaching example. A research classification would normally test a wider predictor set, stronger reference data and a more rigorous spatial validation design.

## 1. Install and import packages

We use:

- **Google Earth Engine** for accessing and processing Sentinel-2, WorldCover and elevation data;
- **geemap** for interactive maps in Colab;
- **pandas** for a small table of Random Forest variable importance;
- **scikit-learn** for the final confusion matrix and accuracy metrics.

The actual Random Forest model is fitted in Earth Engine.

In [ ]:
!pip -q install geemap

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    jaccard_score,
)

## 2. Connect to Google Earth Engine

Earth Engine provides the satellite and environmental datasets and runs the classification on the server.

We initialise the course project once at the beginning of the notebook.

In [ ]:
PROJECT_ID = "rstc2026-earth-engine"

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialised.")

## 3. Define the Adventdalen study area

We use the **same AOI as in Block 1**, so the classification exercise builds directly on the earlier data-access exercise.

Keeping the area small is useful here because:

- the map is easy to inspect visually;
- the classes remain understandable;
- the exercise runs quickly;
- we can focus on the classification workflow rather than on computing time.

The AOI is used consistently for image clipping, sample extraction and the final classified map.

In [ ]:
AOI = ee.Geometry.Rectangle(
    [15.75, 78.16, 16.05, 78.24],
    proj=None,
    geodesic=False,
)

print("AOI: Adventdalen")
print("Bounds: [15.75, 78.16, 16.05, 78.24]")

## 4. Select one good Sentinel-2 acquisition

We use **one summer image**, rather than a seasonal composite.

This makes the logic of the exercise easier to see: all classes are predicted from one clearly defined observation date.

### Why July–August?

For Adventdalen, this period is within the snow-free growing season and is therefore appropriate for a basic vegetation classification.

### Why choose the least cloudy image?

Cloud contamination would introduce pixels that do not represent the land surface. We first sort scenes by the scene-level cloud percentage and select the best candidate.

The scene-level percentage is only a first filter. We still apply a pixel-level cloud mask in the next step.

In [ ]:
START_DATE = "2021-07-01"
END_DATE   = "2021-08-31"

s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

n_scenes = s2_collection.size().getInfo()

if n_scenes == 0:
    raise RuntimeError("No Sentinel-2 scenes found for the selected period.")

best_raw = ee.Image(s2_collection.first())

selected_date = (
    ee.Date(best_raw.get("system:time_start"))
    .format("YYYY-MM-dd")
    .getInfo()
)

scene_cloud = best_raw.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()

print("Candidate scenes:", n_scenes)
print("Selected acquisition:", selected_date)
print(f"Scene-level cloud cover: {scene_cloud:.1f}%")

## 5. Mask cloud, cloud shadow and snow

The Sentinel-2 Scene Classification Layer (SCL) identifies pixels that are unsuitable for land-surface classification.

We remove:

- NoData and defective pixels;
- cloud shadow;
- medium- and high-probability cloud;
- cirrus;
- snow and ice.

### Why do this before training?

The classifier should learn relationships between predictors and real surface classes. If cloudy or shadowed pixels enter the training data, the model can learn artefacts rather than land-surface properties.

The mask is therefore applied before we calculate predictors and before we sample training pixels.

In [ ]:
def mask_sentinel2(image):
    scl = image.select("SCL")

    valid = (
        scl.neq(0)   # NoData
        .And(scl.neq(1))   # saturated / defective
        .And(scl.neq(3))   # cloud shadow
        .And(scl.neq(8))   # cloud, medium probability
        .And(scl.neq(9))   # cloud, high probability
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # snow / ice
    )

    return image.updateMask(valid)


best_clean = mask_sentinel2(best_raw)

## 6. Build the predictor stack

A supervised classifier does not work directly with a visual RGB image. It uses numerical predictor values.

For each pixel we provide:

- **B2, B3, B4, B8** — a small spectral set covering visible and near-infrared reflectance;
- **NDVI** — an interpretable vegetation-sensitive index;
- **Elevation** — an environmental predictor.

### Why these variables?

The four Sentinel-2 bands give the classifier the main spectral information needed to distinguish vegetation, bare ground and water.

NDVI adds an explicit contrast between red and near-infrared reflectance, which is useful for separating vegetated and non-vegetated surfaces.

Elevation provides independent environmental context. It can help where land-cover distribution follows topographic or altitudinal gradients.

### Important point about elevation

The Copernicus DEM is approximately 30 m resolution, while the optical bands are 10 m.

Copernicus GLO-30 is stored in Earth Engine as an **ImageCollection of DEM tiles**. When those tiles are mosaicked, we must explicitly retain a suitable native DEM projection. Otherwise a mosaic can fall back to Earth Engine's very coarse default WGS84 projection, which produces the blocky appearance seen in the previous version.

After fixing the projection, we use bilinear resampling because elevation is a continuous variable.

This does **not** create new 10 m elevation information. It only allows the 30 m DEM to be sampled consistently together with the Sentinel-2 predictors.

In [ ]:
optical = (
    best_clean
    .select(["B2", "B3", "B4", "B8"])
    .multiply(0.0001)
    .clip(AOI)
)

ndvi = (
    optical
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

glo30_collection = (
    ee.ImageCollection("COPERNICUS/DEM/GLO30")
    .filterBounds(AOI)
)

# GLO-30 is stored as an ImageCollection of DEM tiles.
# A plain mosaic of images with different native grids can fall back to
# Earth Engine's default WGS84 / 1-degree projection, which is far too coarse.
# We therefore explicitly keep the native projection of the DEM tiles.
glo30_projection = glo30_collection.first().projection()

elevation = (
    glo30_collection
    .select("DEM")
    .mosaic()
    .setDefaultProjection(glo30_projection)
    .resample("bilinear")
    .rename("elevation")
    .clip(AOI)
)

print("Elevation nominal scale (m):",
      elevation.projection().nominalScale().getInfo())

predictors = (
    optical
    .addBands(ndvi)
    .addBands(elevation)
    .clip(AOI)
)

PREDICTOR_NAMES = ["B2", "B3", "B4", "B8", "NDVI", "elevation"]

print("Predictors:", PREDICTOR_NAMES)

## 7. Inspect the selected image and predictors

Before fitting any model, inspect the input data.

We display:

- true-colour Sentinel-2;
- false-colour Sentinel-2;
- NDVI;
- elevation.

### What are we checking?

We want to confirm that:

- the chosen date looks suitable;
- cloud masking has worked reasonably well;
- NDVI follows expected vegetation patterns;
- the elevation surface covers the AOI correctly.

A classification should not be treated as a black-box operation. Visual inspection of the input data is an important quality-control step.

In [ ]:
def make_map():
    return geemap.Map(
        center=[78.20, 15.90],
        zoom=11,
        basemap="Esri.WorldImagery",
    )


rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.02,
    "max": 0.30,
}

false_colour_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0.02,
    "max": 0.35,
}

ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": ["8c510a", "f6e8c3", "c7eae5", "5ab4ac", "01665e"],
}

elevation_vis = {
    "min": 0,
    "max": 800,
    "palette": ["f7fcf5", "c7e9c0", "74c476", "238b45", "005a32"],
}

Map = make_map()
Map.addLayer(optical, rgb_vis, f"Sentinel-2 RGB — {selected_date}")
Map.addLayer(optical, false_colour_vis, "False colour", False)
Map.addLayer(ndvi, ndvi_vis, "NDVI", False)
Map.addLayer(elevation, elevation_vis, "Elevation", False)
Map.addLayer(AOI, {"color": "yellow"}, "AOI", False)

Map

## 8. Create a simple teaching reference layer

A supervised model needs labelled examples.

For this short exercise we use ESA WorldCover 2021 as a **teaching reference** and merge several detailed WorldCover classes into three broad groups:

- **0 — Vegetation**
- **1 — Bare / sparsely vegetated ground**
- **2 — Water**

Other WorldCover classes are ignored.

### Why keep only three classes?

The purpose is to demonstrate classification clearly within a short exercise.

A more detailed vegetation map would require:

- a more ecologically specific class scheme;
- better reference data;
- enough training samples for every vegetation class;
- stronger validation.

### Why use WorldCover here?

It allows the exercise to be self-contained and reproducible.

For research, reference data should preferably come from fit-for-purpose field observations, expert interpretation or another independent source.

In [ ]:
worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(AOI)
)

reference = worldcover.remap(
    [10, 20, 30, 40, 60, 80, 90, 95, 100],
    [ 0,  0,  0,  0,  1,  2,  0,  0,   0],
    255,
).rename("class")

reference = (
    reference
    .updateMask(reference.neq(255))
    .clip(AOI)
)

class_vis = {
    "min": 0,
    "max": 2,
    "palette": [
        "4CAF50",  # vegetation
        "C9A66B",  # bare / sparse ground
        "3F8FD2",  # water
    ],
}

Map_ref = make_map()
Map_ref.addLayer(optical, rgb_vis, "Sentinel-2 RGB")
Map_ref.addLayer(reference, class_vis, "Teaching reference")
Map_ref.addLayer(AOI, {"color": "yellow"}, "AOI", False)

Map_ref.add_legend(
    title="Reference classes",
    legend_dict={
        "Vegetation": "4CAF50",
        "Bare / sparse ground": "C9A66B",
        "Water": "3F8FD2",
    },
)

Map_ref

## 9. Sample labelled pixels

The reference map covers many pixels, but we do not need to use every one.

We draw a **stratified sample** with up to **150 pixels per class**.

### Why stratified sampling?

Without stratification, the largest class could dominate the sample. Stratified sampling gives each class a clearer representation.

### Why 150 samples per class?

There is no universal correct number.

Here, 150 is a practical compromise:

- large enough to demonstrate the classifier;
- small enough to keep the exercise fast;
- approximately balanced across the three broad classes.

For a research study, sample size should be justified from class variability, spatial coverage and available reference data.

In [ ]:
training_image = (
    predictors
    .addBands(reference)
    .clip(AOI)
)

samples = training_image.stratifiedSample(
    numPoints=150,
    classBand="class",
    region=AOI,
    scale=10,
    geometries=True,
    seed=42,
)

print("Total samples:", samples.size().getInfo())
print("Samples by class:", samples.aggregate_histogram("class").getInfo())

## 10. Separate training and test samples

We now divide the labelled samples into:

- **70% training**
- **30% test**

### What is the training set for?

The model uses the training samples to learn how predictor values relate to the three classes.

For example, it learns combinations of spectral values, NDVI and elevation that tend to occur in vegetation, bare ground or water.

### What is the test set for?

The test samples are kept aside.

They are **not used to fit the Random Forest**. At the end, they provide a simple check of how well the trained model predicts data that were not used during fitting.

### Why 70/30?

It is a common and easy-to-understand teaching split that leaves most samples for fitting while keeping enough samples for evaluation.

It is not presented as the optimal research design.

> In Earth observation, nearby pixels are spatially autocorrelated. A random split can therefore give optimistic accuracy. In a research workflow, use spatial blocks, independent sites or another spatially explicit validation strategy.

In [ ]:
samples = samples.randomColumn("random", seed=42)

train_samples = samples.filter(
    ee.Filter.lt("random", 0.70)
)

test_samples = samples.filter(
    ee.Filter.gte("random", 0.70)
)

print("Training samples:", train_samples.size().getInfo())
print("Test samples reserved for final validation:", test_samples.size().getInfo())

## 11. Train the Random Forest

Training means fitting the model to the labelled training examples.

For every training pixel, the classifier sees:

**B2, B3, B4, B8, NDVI, elevation → known class**

The Random Forest then builds many decision trees that separate the classes using predictor thresholds.

### Why 100 trees?

A Random Forest needs multiple trees so that the final prediction is not dependent on one individual decision tree.

For this small exercise, **100 trees** is sufficient to give a stable ensemble while keeping the model simple and fast.

More trees can be used in research, but increasing the number of trees indefinitely does not automatically improve the classification.

### Why no hyperparameter tuning?

The purpose here is to understand the workflow.

We keep the other Earth Engine Random Forest settings at their defaults so that students can focus on:

- predictors;
- labels;
- model fitting;
- map interpretation;
- validation.

In [ ]:
rf = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42,
).train(
    features=train_samples,
    classProperty="class",
    inputProperties=PREDICTOR_NAMES,
)

print("Random Forest trained.")

### What has the model learned?

At this point, the Random Forest has fitted decision rules from the training examples.

It has **not yet produced the final map**.

The fitted model can now be applied to every valid pixel in the AOI.

The same predictor variables used for training must be available for the pixels we want to classify.

## 12. Optional diagnostic — predictor importance

Random Forest can report how much each predictor contributed to the fitted ensemble.

This can help us ask questions such as:

- Does NDVI contribute strongly?
- Does elevation add useful information?
- Are some spectral bands more informative than others?

### Important caution

Variable importance is a **model diagnostic**, not a causal ecological result.

A high importance value does not mean that the variable causes the class distribution.

Predictors can also share information. For example, red reflectance and NDVI are not independent.

In [ ]:
explanation = rf.explain().getInfo()
importance = explanation.get("importance", {})

importance_df = (
    pd.DataFrame({
        "Predictor": list(importance.keys()),
        "Importance": list(importance.values()),
    })
    .sort_values("Importance", ascending=True)
)

importance_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(
    importance_df["Predictor"],
    importance_df["Importance"],
)
plt.xlabel("Random Forest importance")
plt.title("Predictor importance")
plt.tight_layout()
plt.show()

## 13. Classify the Adventdalen AOI

Now we apply the trained Random Forest to every valid pixel in the predictor stack.

For each pixel, the model evaluates its predictor values and assigns one of the three classes.

The result is clipped to the Adventdalen AOI.

### What is the output?

The visible result is a thematic raster map:

- vegetation;
- bare / sparsely vegetated ground;
- water.

This is the main classification product.

In [ ]:
classified = (
    predictors
    .classify(rf)
    .clip(AOI)
)

In [ ]:
Map_result = make_map()
Map_result.addLayer(
    optical,
    rgb_vis,
    f"Sentinel-2 RGB — {selected_date}",
)

Map_result.addLayer(
    classified,
    class_vis,
    "Random Forest classification",
)

Map_result.addLayer(
    AOI,
    {"color": "yellow"},
    "AOI",
    False,
)

Map_result.add_legend(
    title="Random Forest classes",
    legend_dict={
        "Vegetation": "4CAF50",
        "Bare / sparse ground": "C9A66B",
        "Water": "3F8FD2",
    },
)

Map_result

## 14. Inspect the classification before calculating accuracy

The first evaluation is visual.

Compare the thematic map with the Sentinel-2 image and ask:

1. Does the water class follow rivers, ponds and other dark water surfaces?
2. Does vegetation appear in areas with a plausible vegetation signal?
3. Where does the classifier switch between vegetation and bare / sparse ground?
4. Are there suspicious patterns near cloud-mask gaps or image boundaries?
5. Does elevation appear to help separate surfaces that are spectrally similar?

### Why inspect the map before looking at one accuracy number?

A single number can hide spatially structured errors.

A model may have a high overall accuracy and still fail in ecologically important parts of the landscape.

# 15. Final validation

Validation is the final step in this exercise.

We now use the **held-out test samples** that were not used to fit the model.

For each test sample we have:

- a **reference class**;
- a **predicted class**.

We compare the two.

### Why validate?

Training tells us how to build a model.

Validation tells us how well the fitted model predicts data that were not used for fitting.

Without validation, we only know that a model exists — we do not know whether its classifications are reliable.

In [ ]:
validated = test_samples.classify(rf)

y_true = validated.aggregate_array("class").getInfo()
y_pred = validated.aggregate_array("classification").getInfo()

class_names = [
    "Vegetation",
    "Bare / sparse ground",
    "Water",
]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2],
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)

display.plot()
plt.title("Confusion matrix — held-out test samples")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 16. How to read the confusion matrix

The confusion matrix compares **reference classes** with **predicted classes**.

- Values on the **diagonal** are correct classifications.
- Values away from the diagonal are class confusion.

The matrix is often more informative than one overall number because it shows **which classes are confused with each other**.

For this exercise, pay particular attention to:

- vegetation vs bare / sparse ground;
- whether water is clearly separated;
- whether one class performs noticeably worse than the others.

## 17. Accuracy metrics

We report several complementary metrics because they answer different questions.

### Overall accuracy

**What fraction of all test samples was classified correctly?**

Useful as a broad summary, but it can hide poor performance for individual classes.

### Precision

**When the model predicts a given class, how often is that prediction correct?**

Useful when we want to know whether a mapped class contains many false positives.

### Recall

**Of all reference samples belonging to a class, how many did the model retrieve?**

Useful when missing a class is important.

### F1-score

A balance between precision and recall.

Useful when both false positives and false negatives matter.

### IoU / Jaccard score

Measures the overlap between prediction and reference for each class.

It is widely used for map and segmentation evaluation because it penalises both missed pixels and false assignments.

### Why report class-level metrics?

Vegetation mapping is rarely equally difficult for every class.

A model can have a good overall accuracy while still performing poorly for a smaller or spectrally ambiguous class.

In [ ]:
print(f"Overall accuracy: {accuracy_score(y_true, y_pred):.3f}")
print()

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=class_names,
        digits=3,
        zero_division=0,
    )
)

iou = jaccard_score(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    average=None,
    zero_division=0,
)

print("Class-level IoU")
for name, score in zip(class_names, iou):
    print(f"{name}: {score:.3f}")

## 18. Interpretation

Finish by asking:

1. **Which class is mapped most reliably?**
2. **Which classes are most often confused?**
3. **Did elevation appear useful in the fitted model?**
4. **Are the errors plausible given the image, the reference layer and the class definitions?**
5. **Would the same classifier work equally well elsewhere in Svalbard or in another year?**

### Important validation note

The 70/30 split used here is deliberately simple.

Because neighbouring Earth-observation pixels are spatially autocorrelated, a research study should preferably use:

- spatially separated test areas;
- spatial block cross-validation;
- independent field sites;
- or another validation design matched to the intended area of application.

### Main workflow

**one Sentinel-2 acquisition → spectral + environmental predictors → labelled samples → Random Forest → classified AOI → visual interpretation → final validation**